# RankResume - Document to PostgreSQL Pipeline

This notebook runs local document extraction, structured JSON parsing, validation, PostgreSQL JSONB storage, and dataset export. It does not implement matching or ranking.

## 1. Install dependencies if needed

In [46]:
# Run once in the notebook kernel if imports are unavailable.
# %pip install pdfplumber python-docx psycopg2-binary skill-extractor

## 2. Import the modular pipeline

In [47]:
import sys
import importlib
from pathlib import Path

DB_STORE = Path('/home/vedant-2506/Desktop/Python_ML/Project/DB_Store')
if str(DB_STORE) not in sys.path:
    sys.path.insert(0, str(DB_STORE))
import rankresume_pipeline
importlib.reload(rankresume_pipeline)
from rankresume_pipeline import *

print('Configured resumes path:', RESUMES_DIR)
print('Configured JDs path:', JDS_DIR)
print('Resume extensions:', sorted(RESUME_EXTENSIONS))
print('JD extensions:', sorted(JD_EXTENSIONS))

Configured resumes path: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes
Configured JDs path: /home/vedant-2506/Desktop/Python_ML/Project/Data/jds
Resume extensions: ['.doc', '.docx', '.pdf']
JD extensions: ['.csv', '.doc', '.docx', '.json', '.pdf']


## 3. Verify input folders and local readers

In [48]:
resume_dir, jd_dir = resolve_input_dirs()
print('Using resumes:', resume_dir)
print('Using JDs:', jd_dir)
print('Resume files:', len(files_in(resume_dir)))
print('JD files:', len(files_in(jd_dir)))
print('pdfplumber reader: available')
print('python-docx reader: available')
print('DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered')

Using resumes: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes
Using JDs: /home/vedant-2506/Desktop/Python_ML/Project/Data/jds
Resume files: 69
JD files: 25
pdfplumber reader: available
python-docx reader: available
DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline scans all supported files, checks each filename before extraction, inserts only new documents, and never deletes database rows when source files are removed.

In [49]:
sample = 'C++ C# .NET ASP.NET Node.js React.js Next.js CI/CD Spring Boot REST API PostgreSQL Machine Learning JavaScript Kubernetes Apache Kafka'
print('Technology aliases:', normalize_skills(['C++', 'cpp', 'c plus plus', 'JavaScript', 'js', 'Machine Learning', 'ml']))
print('Discovered skills:', discover_skills(sample))
print('Cleaned sample:', clean_text(sample))
print('Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.')

Technology aliases: ['c++', 'javascript', 'machine learning']
Discovered skills: ['c++', 'c#', '.net', 'node.js', 'react.js', 'next.js', 'ci/cd', 'spring boot', 'restful api', 'postgresql', 'machine learning', 'javascript', 'apache kafka', 'kubernetes']
Cleaned sample: c++ c# .net .net node.js react.js next.js ci/cd spring boot rest api postgresql machine learning javascript kubernetes apache kafka
Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline synchronizes application rows to files currently present in the Data folders and preserves the existing database credentials and JSON schema.

In [50]:
RESET_TABLES = False
connection = connect_db()
print('Verified database:', DB_NAME)
print('Existing public tables:', inspect_tables(connection))
create_schema(connection, reset=RESET_TABLES)
seed_dictionary(connection)
esco_csv = find_esco_skills_csv()
esco_count = import_esco_skills(connection, esco_csv)
load_esco_labels(connection)
print('ESCO CSV:', esco_csv)
print('ESCO records:', esco_count)
print('Schema, technology aliases, and ESCO reference data are ready.')

Verified database: rankresume
Existing public tables: ['ESCO_Skills', 'candidate', 'job_description', 'technology_alias', 'technology_dictionary']


2026-09-24 11:50:14,638 INFO: Imported 13939 ESCO skills from /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv


ESCO CSV: /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv
ESCO records: 13939
Schema, technology aliases, and ESCO reference data are ready.


## 6. Run the complete batch

Each document is handled independently. A failed or unsupported file is logged and does not stop the remaining batch.

In [51]:
summary = process_batch(connection, reset=False)
print_summary(summary)
print('\n========== SAMPLE RESUME JSON ==========')
resume_dataset = json.loads((DATA_ROOT / 'resumes.json').read_text(encoding='utf-8'))
print(json.dumps(resume_dataset['resumes'][0], indent=2, ensure_ascii=False)[:3000])
print('\n========== SAMPLE JD JSON ==========')
jd_dataset = json.loads((DATA_ROOT / 'jds.json').read_text(encoding='utf-8'))
print(json.dumps(jd_dataset['job_descriptions'][0], indent=2, ensure_ascii=False)[:3000])

2026-09-24 11:50:16,425 INFO: Imported 13939 ESCO skills from /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv
2026-09-24 11:50:17,147 INFO: Scanning resumes directory: found 69 supported files
2026-09-24 11:50:17,147 INFO: Scanning JDs directory: found 25 supported files
2026-09-24 11:50:17,148 INFO: [1/69] Checking resume: Aarya_Kale.pdf
2026-09-24 11:50:17,150 INFO: Already exists in database. Skipping: Aarya_Kale.pdf
2026-09-24 11:50:17,151 INFO: [2/69] Checking resume: Abhijeet.pdf
2026-09-24 11:50:17,152 INFO: Already exists in database. Skipping: Abhijeet.pdf
2026-09-24 11:50:17,153 INFO: [3/69] Checking resume: Aditi Bhosure Resume.pdf
2026-09-24 11:50:17,155 INFO: Already exists in database. Skipping: Aditi Bhosure Resume.pdf
2026-09-24 11:50:17,156 INFO: [4/69] Checking resume: Adity_Gowari.pdf
2026-09-24 11:50:17,158 INFO: Already exists in database. Skipping: Adity_Gowari.pdf
2026-09-24 11:50:17,159 INFO: [5/69] Checking resume: Ai

INGESTION SUMMARY

Resumes:
    Files found       : 69
    Already in DB     : 69
    Newly inserted    : 0
    Failed            : 0

JDs:
    Files found       : 25
    Already in DB     : 25
    Newly inserted    : 0
    Failed            : 0

Total files checked  : 94
Total new records    : 0
Total skipped        : 94
Total failed         : 0

Database: rankresume
Candidate records: 69
Job Description records: 25

Resume JSON validation: 0 passed, 0 flagged
JD JSON validation: 0 passed, 0 flagged
Datasets: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes.json and /home/vedant-2506/Desktop/Python_ML/Project/Data/jds.json
ESCO records: 13939

========== SAMPLE RESUME JSON ==========
{
  "resume_file_name": "Aarya_Kale.pdf",
  "candidate_name": "AARYA KALE",
  "email": "kaleaarya704@gmail.com",
  "phone": "8421749236",
  "structured_data": {
    "skill": [
      "html5",
      "css3",
      "react.js",
      "restful api",
      "java",
      "c++",
      "spring boot",
      

## 7. Validate stored values

In [52]:
with connection.cursor() as cursor:
    cursor.execute('SELECT COUNT(*) FROM "ESCO_Skills"')
    print('ESCO_Skills count:', cursor.fetchone()[0])
    cursor.execute('SELECT COUNT(*) FROM candidate')
    print('candidate count:', cursor.fetchone()[0])
    cursor.execute('SELECT COUNT(*) FROM job_description')
    print('job_description count:', cursor.fetchone()[0])
    cursor.execute('''
        SELECT candidate_id, resume_file_name, structured_data
        FROM candidate ORDER BY candidate_id DESC LIMIT 5
    ''')
    for candidate_id, filename, structured in cursor.fetchall():
        errors = validate_resume_json(structured)
        print('resume JSON:', candidate_id, filename, 'PASS' if not errors else errors)
        if not errors:
            assert 'certificate' in structured and 'achievements' not in structured
            assert all(set(project) == {'project_title', 'description'} for project in structured['project'])
    cursor.execute('''
        SELECT jd_id, jd_file_name, structured_data
        FROM job_description ORDER BY jd_id DESC LIMIT 5
    ''')
    for jd_id, filename, structured in cursor.fetchall():
        errors = validate_jd_json(structured)
        print('JD JSON:', jd_id, filename, 'PASS' if not errors else errors)
        if not errors:
            assert 'expected_certificate' in structured and 'expected_extra_skill' not in structured
    current_resume_names = {path.name for path in files_in(resume_dir) if path.suffix.lower() in RESUME_EXTENSIONS}
    current_jd_names = {path.name for path in files_in(jd_dir) if path.suffix.lower() in JD_EXTENSIONS}
    cursor.execute('SELECT resume_file_name FROM candidate')
    stored_resume_names = {row[0] for row in cursor.fetchall()}
    cursor.execute('SELECT jd_file_name FROM job_description')
    stored_jd_names = {row[0] for row in cursor.fetchall()}
    print('application resume scope matches Data:', stored_resume_names == current_resume_names)
    print('application JD scope matches Data:', stored_jd_names == current_jd_names)
    cursor.execute('SELECT raw_text = clean_text FROM candidate ORDER BY candidate_id LIMIT 1')
    print('raw_equals_clean:', cursor.fetchone()[0])


ESCO_Skills count: 13939
candidate count: 69
job_description count: 25
resume JSON: 219 dipak patil.pdf PASS
resume JSON: 218 datascienceresumes016.pdf PASS
resume JSON: 217 Varad Tapray.pdf PASS
resume JSON: 216 VaishnavMankar (2).pdf PASS
resume JSON: 215 Testingresumes012.pdf PASS
JD JSON: 53 06_1to3yr_FrontendReactDeveloper_PixelCraft.docx PASS
JD JSON: 52 03_Fresher_JuniorMLEngineer_VertexAnalytics.docx PASS
JD JSON: 51 02_Fresher_AssociateSoftwareEngineer_Brightleaf.docx PASS
JD JSON: 50 05_1to3yr_fullstack_developer.docx PASS
JD JSON: 49 05_1to3yr_BackendJavaEngineer_Orbitline.pdf PASS
application resume scope matches Data: True
application JD scope matches Data: True
raw_equals_clean: False


In [53]:
print('Running complete parser and database validation...')
resume_results = []
for path in files_in(resume_dir):
    if path.suffix.lower() in RESUME_EXTENSIONS:
        parsed = parse_resume(extract_text(path))
        errors = validate_resume_json(parsed)
        resume_results.append((path.name, errors))
        assert not errors, (path.name, errors)

jd_results = []
for path in files_in(jd_dir):
    if path.suffix.lower() in JD_EXTENSIONS:
        raw, source_title = extract_jd_text(path)
        parsed = parse_jd(raw, source_title or extract_job_title(raw))
        errors = validate_jd_json(parsed)
        jd_results.append((path.name, errors))
        assert not errors, (path.name, errors)

assert parse_jd('Minimum 60% marks required.', 'test')['expected_education'][0]['percentage/mark'] == '60%'
assert parse_jd('Candidates should have at least 60 percent.', 'test')['expected_education'][0]['percentage/mark'] == '60%'
assert parse_jd('Achieved 90% improvement in performance.', 'test')['expected_education'] == []
assert len(parse_resume(extract_text(resume_dir / 'Sanjana_Joshi_Resume.pdf'))['project']) == 2
assert parse_resume(extract_text(resume_dir / 'Shravani_Pilane.pdf'))['certificate']
with connection.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM job_description WHERE structured_data ? 'expected_extra_skills'")
    legacy_jd_count = cursor.fetchone()[0]
    cursor.execute("SELECT structured_data FROM job_description WHERE jd_file_name = 'jd_java_developer.pdf'")
    java_jd = cursor.fetchone()[0]
    cursor.execute("SELECT structured_data FROM job_description WHERE jd_file_name = 'JD Intern_SpringCT.pdf'")
    intern_jd = cursor.fetchone()[0]
    cursor.execute("SELECT structured_data FROM candidate WHERE resume_file_name = 'Sanjana_Joshi_Resume.pdf'")
    sanjana_db = cursor.fetchone()[0]
    cursor.execute("SELECT structured_data FROM candidate WHERE resume_file_name = 'Shravani_Pilane.pdf'")
    shravani_db = cursor.fetchone()[0]
assert legacy_jd_count == 0
assert java_jd['expected_education'][0]['percentage/mark'] == '60%'
assert intern_jd['expected_education'][0]['percentage/mark'] == '60%'
assert len(sanjana_db['project']) == 2
assert shravani_db['certificate']
print('Real resume files validated:', len(resume_results))
print('Real JD files validated:', len(jd_results))
print('Stored JD legacy-field count:', legacy_jd_count)
print('Stored failing-case JSONB: PASS')


Running complete parser and database validation...


2026-09-24 11:50:18,893 INFO: Resume extraction counts: projects=2 certificates_or_achievements=0
2026-09-24 11:50:20,014 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2
2026-09-24 11:50:20,554 INFO: Resume extraction counts: projects=0 certificates_or_achievements=0
2026-09-24 11:50:22,124 INFO: Resume extraction counts: projects=2 certificates_or_achievements=0
2026-09-24 11:50:25,599 INFO: Resume extraction counts: projects=3 certificates_or_achievements=0
2026-09-24 11:50:26,229 INFO: Resume extraction counts: projects=3 certificates_or_achievements=0
2026-09-24 11:50:27,227 INFO: Resume extraction counts: projects=2 certificates_or_achievements=3
2026-09-24 11:50:28,074 INFO: Resume extraction counts: projects=2 certificates_or_achievements=0
2026-09-24 11:50:28,705 INFO: Resume extraction counts: projects=0 certificates_or_achievements=0
2026-09-24 11:50:29,474 INFO: Resume extraction counts: projects=0 certificates_or_achievements=0
2026-09-24 11:50:29,

Real resume files validated: 69
Real JD files validated: 25
Stored JD legacy-field count: 0
Stored failing-case JSONB: PASS


In [54]:
print('Targeted failing-case validation')

sanjana_path = resume_dir / 'Sanjana_Joshi_Resume.pdf'
sanjana = parse_resume(extract_text(sanjana_path))
assert len(sanjana['project']) == 2, sanjana['project']
assert sanjana['project'][0]['project_title'] == 'optimizing minor determinant calculations in ecdlp'
assert sanjana['project'][1]['project_title'] == 'development of numerical analysis libraries in c++'
assert all(len(item['project_title']) < len(item['description']) for item in sanjana['project'])
print('Sanjana_Joshi_Resume.pdf: 2 projects, titles/descriptions separated')

shravani_path = resume_dir / 'Shravani_Pilane.pdf'
shravani = parse_resume(extract_text(shravani_path))
assert shravani['certificate'], shravani
assert any('2nd place' in item for item in shravani['certificate'])
assert any('certif' in item for item in shravani['certificate'])
assert 'achievements' not in shravani
print('Shravani_Pilane.pdf: certificates and achievements extracted:', shravani['certificate'])

for name in ('jd_java_developer.pdf', 'JD Intern_SpringCT.pdf'):
    raw, source_title = extract_jd_text(jd_dir / name)
    jd = parse_jd(raw, source_title or extract_job_title(raw))
    assert set(jd) == {'job_title', 'expected_skills', 'expected_education', 'expected_experience', 'expected_certificate'}
    assert 'expected_extra_skills' not in jd
    assert jd['expected_education'][0]['percentage/mark'] == '60%'
    assert '60%' not in jd['expected_education'][0]['degree']
    print(name, 'degree/mark separated, mark=60%, legacy field absent')

print('Targeted failing cases: PASS')


Targeted failing-case validation


2026-09-24 11:51:26,767 INFO: Resume extraction counts: projects=2 certificates_or_achievements=1


Sanjana_Joshi_Resume.pdf: 2 projects, titles/descriptions separated


2026-09-24 11:51:28,061 INFO: Resume extraction counts: projects=2 certificates_or_achievements=3


Shravani_Pilane.pdf: certificates and achievements extracted: ['(cid:136) won 2nd place trophy in a dancing competition (2025)', '(cid:136) won medals in solo dancing and dandia (2025)', '(cid:136) received certificates in dancing and anchoring (2023, 2024, 2025)']


2026-09-24 11:51:28,532 INFO: JD extraction counts: academic_mark=60% education=1 certificates=0


jd_java_developer.pdf degree/mark separated, mark=60%, legacy field absent


2026-09-24 11:51:29,686 INFO: JD extraction counts: academic_mark=60% education=1 certificates=0


JD Intern_SpringCT.pdf degree/mark separated, mark=60%, legacy field absent
Targeted failing cases: PASS


In [55]:
legacy_jds = []
with connection.cursor() as cursor:
    cursor.execute("SELECT jd_file_name FROM job_description WHERE structured_data ? 'expected_extra_skills'")
    legacy_jds = [row[0] for row in cursor.fetchall()]

migration_result = reprocess_existing_documents(
    connection,
    resume_names=['Sanjana_Joshi_Resume.pdf', 'Shravani_Pilane.pdf'],
    jd_names=legacy_jds,
)
print('Controlled reprocessing:', migration_result)
assert migration_result['resumes'] == 2
assert migration_result['jds'] == len(legacy_jds)


2026-09-24 11:51:30,458 INFO: Resume extraction counts: projects=2 certificates_or_achievements=1
2026-09-24 11:51:31,784 INFO: Resume extraction counts: projects=2 certificates_or_achievements=3
2026-09-24 11:51:31,844 INFO: Controlled document reprocessing complete: {'resumes': 2, 'jds': 0}


Controlled reprocessing: {'resumes': 2, 'jds': 0}
